In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%capture
!pip install datasets transformers evaluate rouge_score accelerate, relplot
!pip install git+https://github.com/google-research/bleurt.git
!pip install --upgrade bitsandbytes # numpy pandas
#!pip install unbabel-comet


In [3]:
!pip -q install relplot evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00


In [4]:
!rm -rf colab_llm_utils

In [5]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1224, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1224 (delta 79), reused 93 (delta 43), pack-reused 1092 (from 1)
Receiving objects: 100% (1224/1224), 799.55 KiB | 4.59 MiB/s, done.
Resolving deltas: 100% (768/768), done.


In [6]:
import colab_llm_utils
from colab_llm_utils import configs

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoConfig, BitsAndBytesConfig, GenerationConfig
from accelerate import infer_auto_device_map, init_empty_weights

In [8]:

import time
import re
import sys
import os
import string
import math

In [9]:
import types
import numpy.core as np_core
import numpy as np

np._core = types.ModuleType("_core")
np._core.__dict__.update(np_core.__dict__)

In [10]:
import torch
from torch.nn import functional as F
import evaluate

In [11]:
dataset = colab_llm_utils.configs.datasets.arc_challenge

model_config = colab_llm_utils.configs.models.qwen3_8b
large_model_config = colab_llm_utils.configs.models.qwen3_8b

#embedding_model_config = colab_llm_utils.configs.models.t5_base
short_name = model_config['model_name'].split('/')[-1]


special_tag = ''
special_tag_large = 'rag'



FOLLOW_UP = True
PTRUE = False
SEMDEC = False

In [12]:
DICT_ANS = dataset['dict_ans']
SELF_CONF = False
SKIP_PROCESSING_SMALL = False
SKIP_PROCESSING_LARGE = False
INDEX_JOIN = True
SKIP_MERGING = True
drive_path = f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/"

In [32]:
LARGE_THINKING = True
if LARGE_THINKING:
  special_tag_large = "thinking"

In [14]:

if PTRUE:
  short_name = short_name + "ptrue"
  special_tag = "p_true"


if SELF_CONF:
  short_name = short_name + "confconf"
  special_tag = "self_conf"


if SEMDEC:
  short_name = short_name + "_semlexdec"

In [16]:
metric_dict = {}
if dataset['task_type'] == 'translation':
  metric_dict['meteor'] = colab_llm_utils.scorers.get_meteor()
  metric_dict['bleurt'] = colab_llm_utils.scorers.get_bleurt()
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()
  #comet = colab_llm_utils.scorers.get_comet()
elif dataset['task_type'] == 'summarization':
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()

# Load Intermediate Data Files

In [17]:
files = os.listdir(drive_path)
files.sort(key=lambda x: x) #name sort
files.reverse()

In [18]:
files.reverse()

In [19]:

small_results_7 = []
results_13 = []
results_70 = []


In [20]:
short_name

'Qwen3-8B'

In [21]:
files

['ARC-Challenge_train+test_Qwen3-8B_0000.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0001.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0002.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0003.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0004.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0005.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0006.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0007.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0008.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0009.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0010.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0011.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0012.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0013.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0014.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0015.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0016.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0017.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0018.pickle',
 'ARC-Challenge_train+test_Qwen3-8B_0019.pickle',


In [22]:
f"{short_name}{special_tag}" + "_"

'Qwen3-8B_'

In [23]:
str(dataset["dataset_name"])

'ARC-Challenge'

In [24]:
file_limit = 100
if not SKIP_PROCESSING_SMALL:
  results_7 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    #print(f)
    if len(results_7) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and f"{short_name}{special_tag}" + "_"  in f:
      print(f"**********************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))
      df = colab_llm_utils.data_processing.process_dataframe(df, dataset, metric_dict, self_conf = SELF_CONF, p_true = PTRUE, thinking = False)
      results_7.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")

  res_7 = colab_llm_utils.data_processing.combine_dataframe(results_7)
  if FOLLOW_UP:
    colab_llm_utils.data_processing.columnize_meta_field(res_7, 'follow_up')
    res_7['def_axis'] = [colab_llm_utils.data_processing.coerce_to_bounded_int(out) for out in res_7["follow_up-text"]]
  res_7.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
else:
  res_7 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))

**********************************reading ARC-Challenge_train+test_Qwen3-8B_0000.pickle
Loading ROUGE metric...


Execution time: 10.5278 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0001.pickle
Execution time: 4.8429 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0002.pickle
Execution time: 3.7682 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0003.pickle
Execution time: 2.7645 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0004.pickle
Execution time: 2.5086 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0005.pickle
Execution time: 2.5543 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0006.pickle
Execution time: 2.5809 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0007.pickle
Execution time: 3.5648 seconds
**********************************reading ARC-Challenge_train+test_Qwen3-8B_0008.pickle
Execution time: 3.1600 seconds
****************

In [33]:
large_tag = large_model_config['model_name'].split('/')[-1] + special_tag_large
if not SKIP_PROCESSING_LARGE:
  results_13 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    if len(results_13) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and large_tag  + "_" in f:
      print(f"********************************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))
      df = colab_llm_utils.data_processing.process_dataframe(df, dataset, metric_dict, self_conf=False, p_true = False, thinking=LARGE_THINKING)
      results_13.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")
  res_13 = colab_llm_utils.data_processing.combine_dataframe(results_13)
  res_13.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
else:
  try:
    res_13 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
  except:
    print("adding empty large.")
    res_13 = res_7[res_7['all_probas'] == False]

********************************************reading ARC-Challenge_train+test_Qwen3-8Bthinking_0000.pickle
Execution time: 4.0601 seconds
********************************************reading ARC-Challenge_train+test_Qwen3-8Bthinking_0001.pickle
Execution time: 3.6392 seconds
********************************************reading ARC-Challenge_train+test_Qwen3-8Bthinking_0002.pickle
missing index 5297
in candidates [304, 279, 1931, 358, 34801, 419, 2441, 86183, 1008, 88616]
Execution time: 2.9233 seconds
********************************************reading ARC-Challenge_train+test_Qwen3-8Bthinking_0003.pickle
missing index 25266
in candidates [432, 279, 1045, 2494, 4344, 1052, 11483, 36043, 28752, 14226]
Execution time: 2.8876 seconds
********************************************reading ARC-Challenge_train+test_Qwen3-8Bthinking_0004.pickle
missing index 9688
in candidates [259, 279, 14195, 25111, 4541, 24611, 45735, 7172, 5042, 44742]
Execution time: 2.9265 seconds
****************************

In [26]:
if not SKIP_MERGING:
  if INDEX_JOIN:
    try:
      res_13 = res_13.drop(["ans"], axis=1)
    except:
      pass
    full_res = pd.merge(res_7, res_13, how='left', left_index=True, right_index=True, suffixes=(None,'_large'))
  else:
    full_res = pd.merge(res_7, res_13.drop(["ans"], axis=1), how='left', left_on='prompts', right_on='prompts', suffixes=(None,'_large'))
  #full_res['bleurt_13b'] = [b[0] for b in full_res['bleurt_13b']]
  if SELF_CONF:
    full_res['self_conf'] = [int(c.split("Confidence: ")[-1].strip()[:-1])/100 for c in full_res['self_conf']]
  full_res.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))
#else:
#  full_res = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))

In [27]:
from datetime import datetime
print(f"all done {datetime.now().strftime("%H:%M:%S")}")

all done 00:59:19


In [28]:
res_7['f1'].mean()

np.float64(0.5045)

In [29]:
res_7['follow_up-text']

,follow_up-text
0,1\n\n
1,2\n\n
2,1\n\n
3,1\n\n
4,1\n\n
...,...
1995,1\n\n
1996,1\n\n
1997,1\n\n
1998,1\n\n


In [30]:
res_7['f1'].mean()

np.float64(0.5045)

In [34]:
res_13['f1'].mean()

np.float64(0.7096)